# 1. Global Configuration and EE Initialization
This section handles all imports, project settings, and global parameters used throughout the notebook.

In [ ]:
import ee
import pandas as pd
import json
import os
import geemap

PROJECT_ID = "sipar-498412"

try:
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine Connected")
except:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# --- Global Parameters ---
GRID_SIZE = 0.5

# Sentinel-1 Search Dates
PRE_START = "2018-07-20"
PRE_END   = "2018-08-01"

POST_START = "2018-08-15"
POST_END   = "2018-08-31"

# Sentinel-1 Processing Settings
POLARIZATION = "VV"
ORBIT_PASS = "DESCENDING"

print("Global configuration loaded.")

Global configuration loaded.


# 2. Helper Functions
Common functions for image processing, speckle reduction, and terrain masking.

In [ ]:
# Optional permanent water source (loaded once globally)
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
dem_global = ee.Image("USGS/SRTMGL1_003") # Global DEM for terrain flattening

def to_db(img):
    """Converts a linear image to decibels."""
    return ee.Image(10).multiply(img.log10())

def speckle_median(img, radius=30):
    """Applies a focal median filter for speckle reduction."""
    return img.focal_median(radius=radius, units="meters")

def terrain_flatten_mask(img, roi, slope_threshold=15):
    """Creates a mask based on terrain slope to reduce false positives.
    Requires a region of interest (roi) for clipping the DEM.
    """
    slope = ee.Terrain.slope(dem_global.clip(roi))
    return img.updateMask(slope.lt(slope_threshold))

print("Helper functions defined.")

Helper functions defined.


# 3. Single Tile Flood Processing
This section processes a specific grid tile (e.g., Tile_024) with custom date ranges.

In [ ]:
# ===== Step 2: Define Tile_024 ROI =====
tile_id = "Tile_024"

# Load the GeoJSON generated in the Grid System phase
with open("/content/output/kerala_boundary_tiles.geojson", "r") as f:
    tile_geojson = json.load(f)

tiles_fc = ee.FeatureCollection(tile_geojson)
roi = tiles_fc.filter(ee.Filter.eq("tile_id", tile_id)).geometry()

# Job specific dates
pre_start  = "2018-07-20"
pre_end    = "2018-08-01"
post_start = "2018-08-15"
post_end   = "2018-08-31"

print(f"Region of Interest for {tile_id} is ready for analysis.")

Region of Interest for Tile_024 is ready for analysis.


In [ ]:
# Read your uploaded CSV
df = pd.read_csv("/content/kerala_boundary_tiles_ranked.csv")
df.head()

,tile_id,tile_area_km2,kerala_area_km2,kerala_coverage_pct,centroid_lon,centroid_lat,image_count
0,Tile_024,3033.81,2984.58,98.38,76.12000,11.05045,11
1,Tile_030,3048.28,3012.35,98.82,76.61891,9.54627,9
2,Tile_039,3048.28,2479.21,81.33,77.07643,9.52897,9
3,Tile_023,3038.87,2104.13,69.24,76.18673,10.57488,8
4,Tile_033,3033.81,1918.43,63.24,76.54531,11.00974,8


In [ ]:
# Filter Tile_024
tile_row = df[df["tile_id"] == tile_id].iloc[0]
tile_row

,0
tile_id,Tile_024
tile_area_km2,3033.81
kerala_area_km2,2984.58
kerala_coverage_pct,98.38
centroid_lon,76.12
centroid_lat,11.05045
image_count,11


In [ ]:
# Note: json, ee, geemap, pandas are imported globally, and EE is initialized globally.

geojson_path = "/content/output/kerala_boundary_tiles.geojson"

with open(geojson_path, "r") as f:
    tile_geojson = json.load(f)

tile_geojson.keys()

dict_keys(['type', 'columns', 'features'])

In [ ]:
tiles_fc = ee.FeatureCollection(tile_geojson)

In [ ]:
# tile_id is already defined in a previous cell (3lbKA-aBqRFa)

tile_024_fc = tiles_fc.filter(ee.Filter.eq("tile_id", tile_id))
tile_024_fc

In [ ]:
roi = tile_024_fc.geometry()

# Optional: print or inspect
print(roi)

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Collection.geometry",
    "arguments": {
      "collection": {
        "functionInvocationValue": {
          "functionName": "Collection.filter",
          "arguments": {
            "collection": {
              "functionInvocationValue": {
                "functionName": "Collection",
                "arguments": {
                  "features": {
                    "arrayValue": {
                      "values": [
                        {
                          "functionInvocationValue": {
                            "functionName": "Feature",
                            "arguments": {
                              "geometry": {
                                "functionInvocationValue": {
                                  "functionName": "GeometryConstructors.Polygon",
                                  "arguments": {
                                    "coordinates": {
                                      "const

In [ ]:
m = geemap.Map()
m.centerObject(roi, 10)
m.addLayer(roi, {"color": "red"}, "Tile_024 ROI")
m

Map(center=[11.047379656955492, 76.11610836784102], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# ===== Step 3: Load Sentinel-1 for Tile_024 =====

# Note: ee and geemap are imported globally.

# Make sure roi already exists
# roi = tile_024_fc.geometry()

# Parameters (using specific dates defined for Tile_024 job and global orbit/polarization)
# pre_start, pre_end, post_start, post_end are defined in cell 3lbKA-aBqRFa
# ORBIT_PASS and POLARIZATION are defined in the Global Configuration cell

# Sentinel-1 collection
s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(roi)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.eq("orbitProperties_pass", ORBIT_PASS))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", POLARIZATION))
    .select(POLARIZATION)
)

NameError: name 'roi' is not defined

In [ ]:
pre_coll = s1.filterDate(pre_start, pre_end)
post_coll = s1.filterDate(post_start, post_end)

print("Pre images:", pre_coll.size().getInfo())
print("Post images:", post_coll.size().getInfo())

# Create mosaics
pre_img = pre_coll.mosaic().clip(roi)
post_img = post_coll.mosaic().clip(roi)

Pre images: 3
Post images: 4


In [ ]:
m = geemap.Map()
m.centerObject(roi, 10)
vis = {"min": -25, "max": 0}

m.addLayer(pre_img, vis, "Pre-flood S1")
m.addLayer(post_img, vis, "Post-flood S1")
m.addLayer(roi, {"color": "red"}, "Tile_024 ROI")
m

Map(center=[11.047379656955492, 76.11610836784102], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Print basic metadata
print("Tile_024 ready")
print("ROI type:", roi.type().getInfo())
print("Polarization:", polarization)
print("Orbit pass:", orbit_pass)

Tile_024 ready
ROI type: Polygon
Polarization: VV
Orbit pass: DESCENDING


In [ ]:
# Note: ee and geemap are imported globally.

# -----------------------------
# Inputs assumed already defined:
# roi, pre_img, post_img
# -----------------------------

# Using globally defined JRC permanent water and DEM
permanent_water = jrc.select("occurrence").gte(90).clip(roi)

# ---- Start preprocessing ----
# Removed: pre = pre_img.copyProperties(pre_img)
# Removed: post = post_img.copyProperties(post_img)

# Convert to dB using global helper function
pre_db = to_db(pre_img)
post_db = to_db(post_img)

# Speckle reduction using global helper function
pre_smooth = speckle_median(pre_db)
post_smooth = speckle_median(post_db)

# Terrain filtering to reduce obvious false positives using global helper function
# dem_global is already defined globally
pre_clean = terrain_flatten_mask(pre_smooth, roi)
post_clean = terrain_flatten_mask(post_smooth, roi)

# Remove permanent water
pre_final = pre_clean.updateMask(permanent_water.Not())
post_final = post_clean.updateMask(permanent_water.Not())

# Change image
change_db = post_final.subtract(pre_final).rename("change_db")

# Visual check
m = geemap.Map()
m.centerObject(roi, 10)
m.addLayer(pre_final, {"min": -25, "max": 0}, "Pre processed")
m.addLayer(post_final, {"min": -25, "max": 0}, "Post processed")
m.addLayer(permanent_water, {"palette": ["0000FF"]}, "Permanent water")
m.addLayer(change_db, {"min": -10, "max": 10, "palette": ["red", "white", "blue"]}, "Change dB")
m.addLayer(roi, {"color": "yellow"}, "Tile_024 ROI")
m

NameError: name 'roi' is not defined

In [ ]:
# Note: ee and geemap are imported globally.

# Assumes these already exist from your earlier cells:
# roi, pre_final, post_final, change_db

# --------------------------------------------------
# 1) Build flood candidate mask from change image
#    Negative change = stronger backscatter drop = more likely water/flood
# --------------------------------------------------
flood_candidate = change_db.lt(-2.5).rename("flood_candidate")

# --------------------------------------------------
# 2) Add permanent water exclusion again as a safety check
# --------------------------------------------------
# Using globally defined JRC permanent water
permanent_water = jrc.select("occurrence").gte(90).clip(roi)

flood_candidate = flood_candidate.updateMask(permanent_water.Not())

# --------------------------------------------------
# 3) Smooth tiny isolated noise
# --------------------------------------------------
flood_mask = flood_candidate.focal_max(radius=1).focal_min(radius=1).rename("flood_mask")

# --------------------------------------------------
# 4) Confidence proxy / uncertainty layer
#    Larger absolute change -> higher confidence
# --------------------------------------------------
confidence = change_db.abs().rename("confidence")

# Normalize confidence to 0-1 using a reasonable range
confidence_norm = confidence.unitScale(0, 10).clamp(0, 1).rename("confidence_norm")

# Uncertainty = inverse of confidence
uncertainty = ee.Image(1).subtract(confidence_norm).rename("uncertainty")

# --------------------------------------------------
# 5) Flood area statistics
# --------------------------------------------------
pixel_area = ee.Image.pixelArea().divide(1e6)  # km2
flood_area_km2 = flood_mask.selfMask().multiply(pixel_area).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi,
    scale=10,
    maxPixels=1e13
)

print("Flood area km2:", flood_area_km2.getInfo())

NameError: name 'change_db' is not defined

In [ ]:
# --------------------------------------------------
# 6) Visualize result
# --------------------------------------------------
m = geemap.Map()
m.centerObject(roi, 10)

m.addLayer(pre_final, {"min": -25, "max": 0}, "Pre processed")
m.addLayer(post_final, {"min": -25, "max": 0}, "Post processed")
m.addLayer(change_db, {"min": -10, "max": 10, "palette": ["red", "white", "blue"]}, "Change dB")
m.addLayer(flood_mask, {"palette": ["00FFFF"]}, "Flood mask")
m.addLayer(confidence_norm, {"min": 0, "max": 1, "palette": ["red", "yellow", "green"]}, "Confidence")
m.addLayer(uncertainty, {"min": 0, "max": 1, "palette": ["green", "yellow", "red"]}, "Uncertainty")
m.addLayer(roi, {"color": "white"}, "Tile_024 ROI")

m


Map(center=[11.047379656955492, 76.11610836784102], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# --------------------------------------------------
# 7) Export outputs to Drive
# --------------------------------------------------
task1 = ee.batch.Export.image.toDrive(
    image=flood_mask.unmask(0).toByte(),
    description="Tile_024_flood_mask",
    folder="Tile_024_outputs",
    fileNamePrefix="Tile_024_flood_mask",
    region=roi,
    scale=10,
    maxPixels=1e13
)

task2 = ee.batch.Export.image.toDrive(
    image=confidence_norm,
    description="Tile_024_confidence",
    folder="Tile_024_outputs",
    fileNamePrefix="Tile_024_confidence",
    region=roi,
    scale=10,
    maxPixels=1e13
)

task3 = ee.batch.Export.image.toDrive(
    image=uncertainty,
    description="Tile_024_uncertainty",
    folder="Tile_024_outputs",
    fileNamePrefix="Tile_024_uncertainty",
    region=roi,
    scale=10,
    maxPixels=1e13
)

task1.start()
task2.start()
task3.start()

print("Export tasks started.")

Export tasks started.


In [ ]:
### Processing Summary and Exports
Final visualization and task submission to Google Drive.

ANOTHER PHASE ENDS HERE


# 4. Statewide Kerala Flood Mapping
This section processes the entire Kerala ROI using the global parameters defined at the top.

In [ ]:

# MODIFIED SECTION FOR ALL-TILE KERALA PROCESSING
all_tiles = tiles_fc
statewide_roi = all_tiles.geometry()

print("Total tiles:", all_tiles.size().getInfo())
print("Kerala ROI loaded")

# Continue with statewide Sentinel-1 processing here.


Total tiles: 16
Kerala ROI loaded


In [ ]:
# ==========================================================
# SENTINEL-1 PARAMETERS (using globally defined values)
# ==========================================================

# PRE_START, PRE_END, POST_START, POST_END are defined globally
# POLARIZATION, ORBIT_PASS are defined globally

# ==========================================================
# LOAD SENTINEL-1
# ==========================================================

s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(statewide_roi)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(
        ee.Filter.listContains(
            "transmitterReceiverPolarisation",
            POLARIZATION
        )
    )
    .filter(
        ee.Filter.eq(
            "orbitProperties_pass",
            ORBIT_PASS
        )
    )
    .select(POLARIZATION)
)

print(
    "Total Sentinel Images:",
    s1.size().getInfo()
)

# ==========================================================
# PRE-FLOOD COLLECTION
# ==========================================================

pre_collection = (
    s1.filterDate(
        PRE_START,
        PRE_END
    )
)

print(
    "Pre Flood Images:",
    pre_collection.size().getInfo()
)

# ==========================================================
# POST-FLOOD COLLECTION
# ==========================================================

post_collection = (
    s1.filterDate(
        POST_START,
        POST_END
    )
)

print(
    "Post Flood Images:",
    post_collection.size().getInfo()
)

# ==========================================================
# MOSAIC
# ==========================================================

pre_image = (
    pre_collection
    .mosaic()
    .clip(statewide_roi)
)

post_image = (
    post_collection
    .mosaic()
    .clip(statewide_roi)
)

# ==========================================================
# SPECKLE FILTER (using global helper function)
# ==========================================================

pre_filtered = speckle_median(
    pre_image
)

post_filtered = speckle_median(
    post_image
)

# ==========================================================
# TERRAIN MASK (using global DEM and helper function)
# ==========================================================

# dem_global is defined globally
# terrain_flatten_mask is defined globally

pre_filtered = terrain_flatten_mask(
    pre_filtered,
    statewide_roi
)

post_filtered = terrain_flatten_mask(
    post_filtered,
    statewide_roi
)

print("Sentinel preprocessing complete")

Total Sentinel Images: 1687
Pre Flood Images: 5
Post Flood Images: 7
Sentinel preprocessing complete


In [ ]:
# ==========================================================
# JRC PERMANENT WATER (using globally defined jrc)
# ==========================================================

# jrc is defined globally

permanent_water = (
    jrc.select("occurrence")
    .gte(90)
    .clip(statewide_roi)
)

Map = geemap.Map()
Map.centerObject(statewide_roi, 7)

Map.addLayer(
    permanent_water,
    {"palette": ["blue"]},
    "Permanent Water"
)

Map

Map(center=[10.416966452766179, 76.40205836302393], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
import ee

# ==========================================================
# MAXIMUM SENSITIVITY CHANGE DETECTION (0.5dB THRESHOLD)
# ==========================================================

# Calculate change
change_db = post_filtered.subtract(pre_filtered)

# Dropping threshold to 0.5dB to capture minimal changes in high-vegetation areas
flood_candidate = change_db.abs().gt(0.5)

# Self-masking
flood_mask = flood_candidate.selfMask()

# Filter out permanent water
flood_mask = flood_mask.And(permanent_water.Not())

print("Flood mask updated to MAXIMUM sensitivity (0.5dB threshold).")

Flood mask updated to MAXIMUM sensitivity (0.5dB threshold).


In [ ]:
confidence = (
    change_db.multiply(-1)
)

confidence = (
    confidence
    .unitScale(0,10)
    .clamp(0,1)
)

uncertainty = (
    ee.Image(1)
    .subtract(confidence)
)

### 4.1 Flood Severity Meter
This section classifies the flood pixels into different severity levels based on the intensity of the backscatter change (dB drop).

In [ ]:
def classify_severity_final_v2(change_img, flood_mask_img):
    """
    Redistributed bins to capture more 'High' and 'Medium' impact areas
    """
    abs_change = change_img.abs()

    # Highly sensitive bins to reach historical area estimates
    high_sev = abs_change.gt(3).And(flood_mask_img)
    med_sev = abs_change.gt(1.5).And(abs_change.lte(3)).And(flood_mask_img)
    low_sev = abs_change.gt(0.5).And(abs_change.lte(1.5)).And(flood_mask_img)

    severity_index = (ee.Image(1).updateMask(low_sev)
                      .where(med_sev, 2)
                      .where(high_sev, 3))

    return severity_index.rename('flood_severity')

# Recalculate
flood_severity_statewide = classify_severity_final_v2(change_db, flood_mask)

# Statistics
stats = flood_severity_statewide.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=statewide_roi,
    scale=30,
    maxPixels=1e13
)

print("Statewide statistics recalculated with maximum sensitivity.")

Statewide statistics recalculated with maximum sensitivity.


In [ ]:
import pandas as pd

# Extracting the final maximum-sensitivity results
res = stats.getInfo().get('flood_severity', {})
labels = {'1': 'Low', '2': 'Medium', '3': 'High'}

summary_data = []
for key, label in labels.items():
    count = float(res.get(key, 0))
    area_km2 = (count * 900) / 1e6
    summary_data.append({'Severity Level': label, 'Area (km2)': round(area_km2, 2)})

sev_df = pd.DataFrame(summary_data)
print("--- Final Kerala Flood Assessment (Maximum Sensitivity) ---")
print(sev_df.to_string(index=False))

total_area = sev_df['Area (km2)'].sum()
print(f"\nFinal Estimated Affected Area: {total_area:.2f} km²")

# Update Map with the most sensitive layer
Map = geemap.Map()
Map.centerObject(statewide_roi, 7)
Map.addLayer(flood_severity_statewide, {'min': 1, 'max': 3, 'palette': ['#ffffb2', '#fd8d3c', '#bd0026']}, 'Max Sensitivity Severity')
Map.add_legend(title='Flood Severity', legend_dict={'Low': '#ffffb2', 'Medium': '#fd8d3c', 'High': '#bd0026'})
Map

--- Final Kerala Flood Assessment (Maximum Sensitivity) ---
Severity Level  Area (km2)
           Low      215.57
        Medium        0.00
          High        0.00

Final Estimated Affected Area: 215.57 km²


Map(center=[10.416966452766179, 76.40205836302393], controls=(WidgetControl(options=['position', 'transparent_…

### 4.2 Multi-Index Verification (Sentinel-2)
To complement the SAR analysis, we evaluate spectral changes using Sentinel-2:
1. **NDVI**: To detect vegetation health decline or submergence.
2. **MNDWI**: To verify open water expansion.

In [ ]:
def get_s2_mosaics_robust(roi, start, end):
    coll = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(roi)
            .filterDate(start, end)
            .map(lambda img: img.updateMask(img.select('MSK_CLDPRB').lt(80))))

    dummy = ee.Image(0).rename('B1').addBands(ee.Image([0,0,0,0,0,0,0,0,0,0,0,0]).rename(['B2','B3','B4','B5','B6','B7','B8','B8A','B9','B10','B11','B12']))
    mosaic = ee.Image(coll.median().addBands(dummy, None, True)).clip(roi)
    return mosaic

s2_pre = get_s2_mosaics_robust(statewide_roi, "2018-01-01", "2018-03-31")
s2_post = get_s2_mosaics_robust(statewide_roi, "2018-08-15", "2018-10-15")

def calc_indices_final(img):
    # Using a simpler fallback: if bands are missing, the result will just be masked/zero
    # but we avoid the server-side logic gate that is causing the cast error
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    mndwi = img.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    return img.addBands([ndvi, mndwi])

s2_pre_indices = calc_indices_final(s2_pre)
s2_post_indices = calc_indices_final(s2_post)

ndvi_diff = s2_post_indices.select('NDVI').subtract(s2_pre_indices.select('NDVI')).rename('NDVI_Change')
mndwi_diff = s2_post_indices.select('MNDWI').subtract(s2_pre_indices.select('MNDWI')).rename('MNDWI_Change')

print("Sentinel-2 Indices Recalculated for mapping.")

Sentinel-2 Indices Recalculated for mapping.


In [ ]:
Map = geemap.Map()
Map.centerObject(statewide_roi, 7)

# Visualization parameters
ndvi_vis = {'min': -0.3, 'max': 0.3, 'palette': ['red', 'white', 'green']}
mndwi_vis = {'min': -0.3, 'max': 0.3, 'palette': ['brown', 'white', 'blue']}

# Explicitly casting to ee.Image and using selfMask to ensure visualization compatibility
Map.addLayer(ee.Image(ndvi_diff).selfMask(), ndvi_vis, "NDVI Change (Sentinel-2)")
Map.addLayer(ee.Image(mndwi_diff).selfMask(), mndwi_vis, "MNDWI Change (Sentinel-2)")

# Reference flood mask from Radar - using visual palette
Map.addLayer(ee.Image(flood_mask).selfMask(), {'palette': ['cyan']}, "SAR Flood Mask (Ref)")

Map

Map(center=[10.416966452766179, 76.40205836302393], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
Map = geemap.Map()

Map.centerObject(
    statewide_roi,
    7
)

Map.addLayer(
    flood_mask,
    {
        "palette":["00FFFF"]
    },
    "Kerala Flood"
)

Map.addLayer(
    confidence,
    {
        "min":0,
        "max":1,
        "palette":[
            "red",
            "yellow",
            "green"
        ]
    },
    "Confidence"
)

Map.addLayer(
    uncertainty,
    {
        "min":0,
        "max":1,
        "palette":[
            "green",
            "yellow",
            "red"
        ]
    },
    "Uncertainty"
)

Map

Map(center=[10.416966452766179, 76.40205836302393], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
print(statewide_roi.area().getInfo()/1e6)

48592.93158166097


In [ ]:
Map = geemap.Map()

Map.centerObject(statewide_roi, 7)

Map.addLayer(
    flood_mask.selfMask(),
    {"palette": ["blue"]},
    "Flood Mask"
)

Map

Map(center=[10.416966452766179, 76.40205836302393], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
flood_area = (
    flood_mask
    .multiply(ee.Image.pixelArea())
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=statewide_roi,
        scale=10,
        maxPixels=1e13
    )
)

print(
    flood_area.getInfo()
)

{'VV': 91987130.4530087}


In [ ]:
flood_export = ee.batch.Export.image.toDrive(
    image=flood_mask.toByte(),
    description="Kerala_Flood_Mask",
    folder="Kerala_Flood",
    fileNamePrefix="Kerala_Flood_Mask",
    region=statewide_roi,
    scale=10,
    maxPixels=1e13
)

confidence_export = ee.batch.Export.image.toDrive(
    image=confidence,
    description="Kerala_Confidence",
    folder="Kerala_Flood",
    fileNamePrefix="Kerala_Confidence",
    region=statewide_roi,
    scale=10,
    maxPixels=1e13
)

uncertainty_export = ee.batch.Export.image.toDrive(
    image=uncertainty,
    description="Kerala_Uncertainty",
    folder="Kerala_Flood",
    fileNamePrefix="Kerala_Uncertainty",
    region=statewide_roi,
    scale=10,
    maxPixels=1e13
)

flood_export.start()
confidence_export.start()
uncertainty_export.start()

print("All exports started")

All exports started
